# End-to-end walkthrough

> Train an explainable xAAEnet on a binary image task, then relate latent codes to interpretable feature scores.

This walkthrough ties together the main pieces of the library:

1. **`AAE`** — built-in explainable model, or **`EncoderWithAAEBlocks`** for your own encoder;
2. **`train_xaaenet`** — adversarial → autoencoder → classifier training;
3. **`compute_feature_score_table`** — pixel-level feature scores;
4. **`run_pls_feature_figures`** — alignment panels and importance ranking.

We use the fastai **Oxford-IIIT Pet** dataset as a **cat vs dog** binary task. Adjust paths, class names, and training lengths for your project.


## Setup

Install the package, then run the sections below. Each step imports what it needs from `tell_me_why` (or fastai for the dataset):

```bash
pip install tell-me-why
```


## 1. Binary data: cats vs dogs (PETS)

Each image file name starts with the breed. fastai's usual rule labels **cats** when the breed name starts with an uppercase letter and **dogs** otherwise.

Sample images from the Oxford-IIIT Pet dataset used in this walkthrough:


In [ ]:
#| echo: false
#| exec_doc
#| fig-width: 6.5
#| fig-height: 2.6
from pathlib import Path

from fastai.vision.all import URLs, get_image_files, untar_data
from PIL import Image
import matplotlib.pyplot as plt

pets_path = untar_data(URLs.PETS) / "images"
pet_images = list(get_image_files(pets_path))[:8]

fig, axes = plt.subplots(2, 4, figsize=(8, 2.6))
for ax, image_path in zip(axes.flat, pet_images):
    ax.imshow(Image.open(image_path))
    ax.set_title(image_path.stem[:14], fontsize=7)
    ax.axis("off")
plt.tight_layout(pad=0.4)


## 2. Train an explainable model

We use the built-in **`AAE`** (ResNet-34 encoder + xAAEnet blocks). For a **custom encoder**, use **`EncoderWithAAEBlocks`** instead (documented under *Add xAAEnet Blocks to a User Encoder*).

Build fastai dataloaders on PETS (160×160, cat vs dog), then run `train_xaaenet` (three phases; increase `epochs_*` for real runs).


### Training dataloaders

For `train_xaaenet`, we label **cats** vs **dogs** from the file name (uppercase breed → cat) and resize to **160×160**.


In [ ]:
#| eval: false
from fastai.vision.all import (
    CategoryBlock,
    DataBlock,
    ImageBlock,
    RandomSplitter,
    Resize,
)

def pet_species(path):
    """Cat breeds start with an uppercase letter in the PETS file names."""
    return "cat" if path.name[0].isupper() else "dog"

# Subsample for a quicker first run (remove [:400] for the full dataset)
pet_items = list(get_image_files(pets_path))[:400]

dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=lambda _: pet_items,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=pet_species,
    item_tfms=Resize(160),
)

dls = dblock.dataloaders(pets_path, bs=16, num_workers=0)
dls.vocab, len(dls.train_ds), len(dls.valid_ds)


Choose which species is **class A** (`target_score = 1.0`). With `pet_species`, the vocabulary is `cat` and `dog`:

In [ ]:
#| eval: false
class_a_name = "cat"
class_b_name = "dog"
class_a_name, class_b_name, dls.vocab

In [ ]:
#| eval: false
from pathlib import Path

from tell_me_why.model_aae import AAE
from tell_me_why.training import train_xaaenet

models_dir = Path("walkthrough_models")
models_dir.mkdir(exist_ok=True)

model = AAE(input_size=160, input_channels=3, encoding_dims=128, classes=2)

learn = train_xaaenet(
    model,
    dls,
    epochs_adv=1,
    epochs_ae=1,
    epochs_classif=2,
    models_dir=models_dir,
    save_tsne=False,
    save_pls=False,
    extract_latent=True,
    show_latent_figures=False,
)


## 3. Validation latent codes `z` and binary targets

Collect one latent row per validation image by running the trained model on the validation dataloader. Row order matches `learn.dls.valid` (no shuffling on the validation set).

**Do not reorder** images before the feature table in the next step.


In [ ]:
#| eval: false
import numpy as np
import torch


@torch.no_grad()
def latent_and_targets_from_split(learn, ds_idx=1):
    learn.model.eval()
    z_parts, y_parts = [], []
    for xb, yb in learn.dls[ds_idx]:
        learn.model(xb)
        z_parts.append(learn.model.z.detach().cpu())
        y_parts.append(yb.cpu())
    z = torch.cat(z_parts).numpy()
    targets = torch.cat(y_parts).view(-1).numpy()
    return z, targets

z_val, targets_val = latent_and_targets_from_split(learn, ds_idx=1)
target_score = (targets_val == learn.dls.vocab.o2i[class_a_name]).astype(np.float64)
z_val.shape, target_score.mean()


In [ ]:
#| eval: false
# Same row order as z_val / target_score
image_paths = [str(p) for p in learn.dls.valid.items]
len(image_paths), image_paths[0]


## 4. Feature score table

`compute_feature_score_table` turns each image path into numeric cues (brightness, color, texture, …). Row order must match `z_val` on the validation split.

The table below uses the same PETS sample paths as above. After training, score the full validation set.


In [ ]:
from pathlib import Path

from tell_me_why.feature_scores import compute_feature_score_table


def format_feature_table(df):
    """Drop full paths; show file names first (same layout as the Feature scores example)."""
    out = df.copy()
    out["image_name"] = out["Source_File_Path"].map(lambda path: Path(path).name)
    out = out.drop(columns="Source_File_Path")
    return out[["image_name", *[col for col in out.columns if col != "image_name"]]]


In [ ]:
#| exec_doc
pets_scores = compute_feature_score_table(
    pet_images,
    score_names=[
        "brightness",
        "variance",
        "redness_dominance",
        "symmetry_error",
        "fft_high_frequency_ratio",
    ],
    on_error="raise",
)
format_feature_table(pets_scores).round(4)


### Scores on the validation split

After training, build the table on **every** validation image (all default feature columns). Use the same paths as in step 3, in the same order as `z_val`.


In [ ]:
#| eval: false
df_features = compute_feature_score_table(image_paths)
format_feature_table(df_features).round(4)


## 5. Classification interpretation figures

Compare latent axis PLS1 to each feature. The section *Reading the alignment panels* and *Reading the importance ranking* on the **Classification Interpretation** page explains how to read the figures.


In [ ]:
#| eval: false
from tell_me_why.visualization import run_pls_feature_figures

out = run_pls_feature_figures(
    z_val,
    target_score,
    df_features,
    target_label=class_a_name,
    mask_positive=target_score.astype(bool),
    positive_label=class_a_name,
    negative_label=class_b_name,
    save_dir=models_dir / "interpretation",
    show=False,
)

out["importance_rank"][:5]


## Reading the results (short checklist)

- **Importance ranking** — features at the top have the strongest signed r² with PLS1 toward class A; near-zero bars were weakly aligned on this axis.
- **Alignment panels** — diagonal green line + separated grey/blue clouds suggest a pixel cue that co-varies with the latent decision axis; a flat line suggests little linear link.
- **Causality** — alignment is a **comparison** between latent space and hand-crafted scores, not a proof of what neurons implement.

## Next steps

- Train longer and on the full PETS split (or your own binary dataset).
- Swap `AAE` for `EncoderWithAAEBlocks` when you bring a custom encoder.
- Reuse the same `z` + `df_features` + `run_pls_feature_figures` pipeline after any binary xAAEnet training run.


In [ ]:
#| hide
#| eval: false
import numpy as np
import pandas as pd
from tell_me_why.visualization import run_pls_feature_figures
rng = np.random.default_rng(0)
n = 48
z = rng.normal(size=(n, 32))
target_score = rng.integers(0, 2, size=n).astype(float)
cols = ["brightness", "variance", "redness_dominance"]
df = pd.DataFrame({c: rng.normal(size=n) for c in cols})
out = run_pls_feature_figures(
    z, target_score, df, feature_columns=cols, target_label="cat", show=False
)
assert len(out["importance_rank"]) == 3
